# Pipeline stage 3: characterization of phage-encoded acetyltransferase Map

### Transcriptomic analysis of LUZ19 infection

In our research, we have characterized LUZ19 gp13/Map through the use of overexpression assays. To get a better idea of its biological relevance during phage infection, we will reanalyze existing transcriptomics data [(Brandao *et al*, 2021)](https://doi.org/10.1080/15476286.2020.1870844), to identify the timing and extent of Map expression. This reanalysis is necessary, as the original analysis did not use the *map* gene boundaries as described by Lavigne *et al*. 

For this reanalysis, we will use the transcriptomic analysis pipeline "phage_host_dual_transcriptomics" v.1.0 from [(Wolfram-Schauerte *et al* 2026)](https://doi.org/10.64898/2026.03.30.715471). To run this notebook correctly, the pipeline should be installed as described on [GitHub](https://github.com/Integrative-Transcriptomics/phage_host_dual_transcriptomics), and the corresponding conda environment should be created. This notebook creates the scripts to prepare all necessary files to run this pipeline, and then analyze the output of the pipeline. The analysis is strongly based on the original "downstream_processing.ipynb" notebook.   

**Goals:**
* Prepare necessary files for transcriptomics analysis on the HPC.
* Analyze output from the pipeline.

**Requires:**
* "phage_host_dual_transcriptomics" pipeline v1.0 by [(Wolfram-Schauerte *et al* 2026)](https://doi.org/10.64898/2026.03.30.715471) and the accompanying downstream processing file tools.py
* transcriptomic data from [(Brandao *et al*, 2021)](https://doi.org/10.1080/15476286.2020.1870844): the code in this notebook actually creates scripts to automatically download and store the sequencing data hosted on SRA
* transcriptomic metadata file SraRunTable.csv from [(Brandao *et al*, 2021)](https://doi.org/10.1080/15476286.2020.1870844) stored in input/: metadata file downloaded from the [SRA](https://www.ncbi.nlm.nih.gov/Traces/study/?acc=PRJNA681237)
* reference genome (annotation) files PAO1.fa, PAO1.gff, LUZ19.fa and LUZ19_edited.gff, stored in input/: these files were downloaded from NCBI datasets, with the following identifiers: NC_002516.2 (PAO1.fa), GCF_000006765.1 (PAO1.gff), NC_010326.1 (LUZ19.fa), GCF_000879115.1 (LUZ19_edited.gff). The LUZ19 annotation file was manually edited, to update the start coordinate of *map* to 7021 (for both the CDS and gene), and was hence renamed LUZ19_edited.gff to reflect this deviation from the data downloaded from NCBI

**Generates:**
* in folder input/:
    * sra_fetch.slurm: jobscript to download transcriptomic data on HPC
    * slurm-60056488.out: output file for download script
    * SRRXXXXX_1.fastq.gz and SRRXXXXX_2.fastq.gz for identifiers SRR13160327 up to SRR13160338: gzipped sequencing data
* txome_pipeline.slurm: jobscript to run the "phage_host_dual_transcriptomic" Nextflow pipeline on the HPC
* slurm-60059173.out: output file for Nextflow pipeline
* in folder output/:
    * pipeline output, distributed over folder alignments/, cutadapt/, countData/, fastQC/ and multiQC/ (see  "phage_host_dual_transcriptomics" docs for more information on the different folders and files)

#### General settings, imports, variables and environments

Conda environment: viral_act_txomics

Created with `conda create -n viral_act_txomics python=3.10`. Then installed  `jupyter notebook`, `pandas` through conda (command `conda install`), and `seaborn`, `scikit-learn` through pip (command below).

In [1]:
!pip install seaborn

In [2]:
!pip install scikit-learn

In [3]:
!conda list --explicit

# This file may be used to create an environment using:
# $ conda create --name <env> --file <this file>
# platform: win-64
# created-by: conda 24.11.3
@EXPLICIT
https://conda.anaconda.org/conda-forge/noarch/ca-certificates-2026.5.20-h4c7d964_0.conda
https://conda.anaconda.org/conda-forge/win-64/onemkl-license-2026.0.0-h57928b3_908.conda
https://conda.anaconda.org/conda-forge/noarch/python_abi-3.10-8_cp310.conda
https://conda.anaconda.org/conda-forge/noarch/tzdata-2025c-hc9c84f9_1.conda
https://conda.anaconda.org/conda-forge/win-64/ucrt-10.0.26100.0-h57928b3_0.conda
https://conda.anaconda.org/conda-forge/win-64/winpty-0.4.3-4.tar.bz2
https://conda.anaconda.org/conda-forge/win-64/libwinpthread-12.0.0.r4.gg4f2fc60ca-h57928b3_10.conda
https://conda.anaconda.org/conda-forge/win-64/vcomp14-14.51.36231-h1b9f54f_37.conda
https://conda.anaconda.org/conda-forge/win-64/vc14_runtime-14.51.36231-h1b9f54f_37.conda
https://conda.anaconda.org/conda-forge/win-64/vc-14.5-h1b7c187_37.conda
https://conda

In [5]:
# imports 
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from tools import *

In [6]:
# clear reference to different directories
pipeline_map_dir = os.getcwd()
master_dir = os.path.abspath(os.path.join(pipeline_map_dir, os.pardir, os.pardir))

In [ ]:
# creating a directory for all the data we will generate
os.mkdir(os.path.join(pipeline_map_dir, "transcriptomics_analysis"))

#### Goal 1 : prepare necessary files for transcriptomics analysis on the HPC

To be able to run the transcriptomics analysis pipeline from [(Wolfram-Schauerte *et al* 2026)](https://doi.org/10.64898/2026.03.30.715471), we first have to obtain the transcriptomics data from [(Brandao *et al*, 2021)](https://doi.org/10.1080/15476286.2020.1870844). Let's write a jobscript to do that. 

In [ ]:
# creating a directory for all the transcriptomics data we will store
os.mkdir(os.path.join(pipeline_map_dir, "transcriptomics_analysis", "input"))

In [7]:
# overview of the SRA identifiers for the transcriptomics data of LUZ19 infection in LB medium
    # based on the metadata from https://www.ncbi.nlm.nih.gov/Traces/study/?acc=PRJNA681237
sra_ids = [f"SRR131603{i}" for i in range(27, 39)]

In [8]:
# base script for fetching the transcriptomics data from the SRA database using the SRA-Toolkit on the HPC
script_content = '''#!/bin/bash

#SBATCH --cluster=wice
#SBATCH --nodes=1 --ntasks-per-node=8
#SBATCH --time=12:30:00

cd $SLURM_SUBMIT_DIR

module load cluster/wice/batch
module load SRA-Toolkit/3.0.5-gompi-2021a 

'''

In [9]:
# now add in the downloading commands for each of the SRA identifiers
for sra_id in sra_ids:
    script_content += f"prefetch {sra_id}\n"
    script_content += f"fastq-dump --split-files --gzip  {sra_id}\n"

In [10]:
script = os.path.join(pipeline_map_dir, "transcriptomics_analysis", "input", "sra_fetch.slurm")
with open(script, "w") as file:
    print(script_content, file = file)

Now we can put that script on the HPC and use it to download the correct transcriptomics data. 

Next, we can create the jobscript for running the ["phage host dual transcriptomics" Nextflow pipeline](https://github.com/Integrative-Transcriptomics/phage_host_dual_transcriptomics).  

In [11]:
script_content = '''#!/bin/bash

#SBATCH --cluster=wice
#SBATCH --nodes=1 --ntasks-per-node=16
#SBATCH --time 06:00:00

cd $SLURM_SUBMIT_DIR

module load cluster/wice/batch
module load Nextflow/25.04.8

source /data/leuven/331/vsc33164/miniconda3/bin/activate

nextflow run pipeline --reads "$VSC_SCRATCH/txome/input/*_{1,2}.fastq.gz" --pairedEnd --hostGenome "$VSC_SCRATCH/txome/input/PAO1.fa" --hostGFF "$VSC_SCRATCH/txome/input/PAO1.gff" --phageGenome "$VSC_SCRATCH/txome/input/LUZ19.fa" --phageGFF "$VSC_SCRATCH/txome/input/LUZ19_edited.gff" --outputDir "$VSC_SCRATCH/txome/output" --countFeature "gene" --featureIdentifier "ID" --conda_path "$VSC_DATA/miniconda3/envs" 
'''

In [12]:
script = os.path.join(pipeline_map_dir, "transcriptomics_analysis", "txome_pipeline.slurm")
with open(script, "w") as file:
    print(script_content, file = file)

This script can then also be uploaded to the HPC. 

To run the pipeline, the GitHub repo containing the pipeline should be present on your HPC. Place the jobscript wihtin the main folder of the "phage_host_dual_transcriptomics" before running it. 

#### Goal 2 : analyze output from the pipeline

Now that we have run the "phage_host_dual_transcriptomics" pipeline, we can analyze its results, to assess the timing and extent of Map expression during LUZ19 infection. We will do this by making use of the code from the accompanying "downstream_processing.ipynb" notebook, which is copied here. 

First, we load the necessary datasets:

In [ ]:
# file paths to necessary files for the transcriptomics analysis
bulkPath = os.path.join(pipeline_map_dir, "transcriptomics_analysis", "output", "countData", "countData.tsv") # output from nf pipeline
metaPath = os.path.join(pipeline_map_dir, "transcriptomics_analysis", "input", "SraRunTable.csv") # metadata from SRA
gffPath = os.path.join(pipeline_map_dir, "transcriptomics_analysis", "output", "alignments", "dualGenome.gff3") # output from nf pipeline

In [14]:
# load data
df_initial = pd.read_csv(bulkPath, sep = '\t', comment='#', index_col=0)
metadata = pd.read_csv(metaPath)

Next, we can format the data:

In [15]:
# Match GSM IDs and SampleNames inferred from GEO
sampleDict = {'GSM4948299': '0_R1', 'GSM4948300': '0_R2', 'GSM4948301': '0_R3',
              'GSM4948302': '5_R1', 'GSM4948303': '5_R2', 'GSM4948304': '5_R3',
              'GSM4948305': '10_R1', 'GSM4948306': '10_R2', 'GSM4948307': '10_R3',
              'GSM4948308': '15_R1', 'GSM4948309': '15_R2', 'GSM4948310': '15_R3',
}

In [ ]:
# use the annotateData function to add the sample names to the metadata dataframe
metadataFull = annotateData(metadata, sampleDict)

In [17]:
# make sure the column names reflect the different timepoints and replicates, and take the relevant subset
df = changeColnames(df_initial.iloc[:,5:df_initial.shape[1]], metadataFull)
df = df[['0_R1', '0_R2', '0_R3', '5_R1', '5_R2', '5_R3', '10_R1', '10_R2', '10_R3', '15_R1', '15_R2', '15_R3']]


Now, we do an in silico rRNA depletion:

In [19]:
# Load gff3 and split into genes and CDS dfs
gff3 = pd.read_csv(gffPath, sep='\t', header = None, skiprows = 7)
gff3.columns=["seq_id", "source", "type", "start", "end", "phase", "strand", "score", "attributes"]
gff3_genes = gff3.loc[gff3["type"] == 'gene']

# Column formating for genes
gff3_genes = gff3_genes.reset_index(drop=True)
dct_genes = gff3_genes["attributes"].str.split(';').apply(lambda items: dict(item.split('=', 1) for item in items if '=' in item))
cols_to_keep = ['ID', 'Name', 'gbkey', 'gene_biotype', 'locus_tag', 'gene']
gff3_genes = pd.concat([gff3_genes, pd.json_normalize(dct_genes)[cols_to_keep]], axis=1)

# Generate locus_tag, product dictonary over all different feature types
attrs = gff3["attributes"].str.split(";", expand=True)
attrs_dicts = attrs.apply(lambda row: {item.split("=")[0]: item.split("=")[1] for item in row if "=" in str(item)}, axis=1)
attrs_df = pd.json_normalize(attrs_dicts)
attrs_df = attrs_df.dropna(subset=["locus_tag", "product"])
locus_product_dict = dict(zip(attrs_df["locus_tag"], attrs_df["product"]))

# Add gene product, if not stated in gff3, fill with gene_biotype
gff3_genes["product"] = gff3_genes["locus_tag"].map(locus_product_dict)
gff3_genes["product"] = gff3_genes["product"].fillna(gff3_genes["gene_biotype"])

# If gene = NA, take from ID column
gff3_genes["gene"] = gff3_genes["gene"].fillna(gff3_genes["ID"])

# Drop attributes column
gff3_genes = gff3_genes.drop(["attributes"], axis=1)

In [21]:
# Load ggf3 file
gff3 = pd.read_csv(gffPath, sep='\t', header = None, skiprows = 7)
gff3 = gff3.loc[gff3.iloc[:,2] == 'gene']

# Format some new columns
gff3['ID'] = pd.DataFrame(gff3.iloc[:,8].str.split('ID=', expand = True)).iloc[:,1].str.split(';', expand = True).iloc[:,0]
gff3['GeneType'] = pd.DataFrame(gff3.iloc[:,8].str.split('gene_biotype=', expand = True)).iloc[:,1].str.split(';', expand = True).iloc[:,0]
gff3['Symbol'] = pd.DataFrame(gff3.iloc[:,8].str.split('gene=', expand = True)).iloc[:,1].str.split(';', expand = True).iloc[:,0]

# Add entity host and phage
gff3['Entity'] = np.where(gff3[0] == 'NC_010326.1', 'phage', 'host')
gff3.index = gff3['ID']
rRNAs = gff3.loc[gff3['GeneType'] == 'rRNA', 'ID'].tolist()

In [23]:
df_norRNAs = rRNAdepletion(df,rRNAs)

Next, we can start the read count normalization process:

In [24]:
# Function to fill in missing symbols by geneid.

def fillSymbols(df):
    df_new = df.copy()
    index = df.index.to_list()
    for i in range(0,df.shape[0]):
        if (df.iloc[i,-1:].values == None):
            df_new.iloc[i,-1:] = index[i]
    return df_new

In [ ]:
# get transcripts per million (TPM) values for all genes, and add the gene symbols and entity information to the dataframe
tpms = TPM(df_norRNAs, df_initial, 0.5)
tpms['Entity'] = gff3.loc[sorted(tpms.index.to_list()), 'Entity']
tpms['Symbol'] = gff3.loc[sorted(tpms.index.to_list()), 'Symbol']

tpms = fillSymbols(tpms)
tpms = make_unique_with_index(tpms)

In [34]:
# now calculate mean and standard deviation of TPMs for each timepoint, and add the gene symbols and entity information to the dataframe
TPMmeans, TPMsds = getMeanSD(tpms[['0_R1', '0_R2', '0_R3', '5_R1', '5_R2', '5_R3', '10_R1', '10_R2', '10_R3', '15_R1', '15_R2', '15_R3']])
TPMmeans = TPMmeans[['0', '5', '10', '15']]
TPMmeans[['Entity', 'Symbol']] = tpms[['Entity', 'Symbol']]

c:\Users\hanne\OneDrive\Documenten\GitHub\ACES-AcT\pipeline\3_map\tools.py:111: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  means[mean] = np.mean(df.iloc[:,indices], axis = 1)
c:\Users\hanne\OneDrive\Documenten\GitHub\ACES-AcT\pipeline\3_map\tools.py:112: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sds[sd] = np.std(df.iloc[:,indices], axis = 1)
c:\Users\hanne\OneDrive\Documenten\GitHub\ACES-AcT\pipeline\3_map\tools.py:111: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a 

Finally, I can use this data to extract the rank of the *gp13* transcript, to see in which stage of infection it is expressed (abundantly):

In [40]:
timepoints = ['0', '5', '10', '15']

for tp in timepoints:
    ranked = TPMmeans[TPMmeans['Entity'] == 'phage'].sort_values(by=tp, ascending=False).reset_index(drop=True)
    rank = ranked[ranked['Symbol'] == 'gp13'].index[0] + 1
    value = ranked.loc[rank-1, tp]
    print(f"Time {tp}: rank = {rank}, TPM = {value}")


Time 0: rank = 31, TPM = 0.917779475806876
Time 5: rank = 6, TPM = 9962.91352776926
Time 10: rank = 16, TPM = 15259.681974153391
Time 15: rank = 42, TPM = 2991.786817494251


In [42]:
timepoints = ['0', '5', '10', '15']

for tp in timepoints:
    ranked = TPMmeans.sort_values(by=tp, ascending=False).reset_index(drop=True)
    rank = ranked[ranked['Symbol'] == 'gp13'].index[0] + 1
    value = ranked.loc[rank-1, tp]
    print(f"Time {tp}: rank = {rank}, TPM = {value}")
print(len(TPMmeans))


Time 0: rank = 5491, TPM = 0.917779475806876
Time 5: rank = 17, TPM = 9962.91352776926
Time 10: rank = 19, TPM = 15259.681974153391
Time 15: rank = 48, TPM = 2991.786817494251
5719
